# Supplementary Figure S7 — fitted parameters and chirp tuning

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS = ROOT / "saved_results"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.style.use(ROOT / "notebooks/ncb.mplstyle")
plt.close("all")
np.random.seed(0)


In [ ]:
import csv
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

data = np.load(SAVED_RESULTS / 'results_data.npz', allow_pickle=True)
with np.load(SAVED_RESULTS / 'fits_best_available.npz', allow_pickle=True) as archive:
    fits = archive['fits']
fit_by_species = {fit['species']: fit for fit in fits}
with (ROOT / 'data/species_names.csv').open(newline='') as source:
    aliases = {row['species_code']: row['figure_code'] for row in csv.DictReader(source)}
species = data['species']
labels = [aliases.get(code, code) for code in species]

In [ ]:
parameters = [
    ('inh_mult', 'Inhibitory filter', '#e08080', 4),
    ('ada_mult', 'Divisive normalization', '#6b9bcb', 1.5),
]
features = [
    ('best_cdc', 'Preferred duty cycle', 1),
    ('max_cdc_resp', 'Duty-cycle high-pass', 1),
    ('best_cper', 'Preferred chirp period [s]', 1000),
    ('max_cper_resp', 'Period high-pass', 1),
]
offsets = {
    (0, 2): {'BIM': (8, 12), 'STA': (-7, 6), 'LIN': (8, -8),
             'LIG': (-5, 5), 'TUL': (-8, -8), 'FIR': (5, -12),
             'ARM': (-5, -13), 'TEX': (4, -13)},
    (0, 3): {'BIM': (-5, 6), 'STA': (2, 15), 'PER': (4, -12),
             'LON': (-5, -12), 'LIG': (-5, 9), 'FIR': (5, -12),
             'ARM': (-5, -14), 'TEX': (0, 9), 'REG': (5, -14)},
    (1, 1): {'ARM': (-5, -12), 'LIG': (-5, 7), 'TUL': (5, -10),
             'STA': (-5, 6), 'LIN': (5, -9), 'FIR': (-5, 6), 'LON': (5, -12)},
    (1, 2): {'STA': (-8, 6), 'LIN': (6, 7), 'BIM': (6, -10),
             'LIG': (-6, 7), 'TUL': (6, -3), 'FIR': (6, -12)},
    (1, 3): {'STA': (-5, 6), 'LIN': (5, -10), 'BIM': (-5, -10),
             'LON': (-5, -12), 'FIR': (6, -7), 'TUL': (5, -10)},
}
fig, axes = plt.subplots(2, 4, figsize=(12.8, 5.2), sharex='col',
                         sharey='row', constrained_layout=True)
for row, (parameter, row_label, color, ymax) in enumerate(parameters):
    y = np.array([fit_by_species[code][parameter] for code in species])
    for column, (feature, xlabel, scale) in enumerate(features):
        ax = axes[row, column]
        x = np.asarray(data[feature]) / scale
        ax.scatter(x, y, s=18, color=color)
        for label, xi, yi in zip(labels, x, y):
            dx, dy = offsets.get((row, column), {}).get(
                label, (-4 if xi > (.85 * (2 if column == 2 else 1)) else 4,
                        -8 if yi > .85 * ymax else 5))
            ax.annotate(label, (xi, yi), xytext=(dx, dy),
                        textcoords='offset points', fontsize=7,
                        ha='right' if dx < 0 else 'left')
        rho, pvalue = spearmanr(x, y)
        ax.set_title(f'ρ = {rho:.2f}, p = {pvalue:.2g}', fontsize=9)
        ax.text(-.12, 1.04, chr(65 + row * 4 + column),
                transform=ax.transAxes, weight='bold', fontsize=14)
        ax.set(xlim=(0, 2 if column == 2 else 1), ylim=(0, ymax))
        ax.spines[['top', 'right']].set_visible(False)
        if row == 1:
            ax.set_xlabel(xlabel)
        if column == 0:
            ax.set_ylabel('Inh. filter multiplier' if row == 0 else 'Div. norm. multiplier', fontsize=9)
fig.savefig(FIGURES / 'fig_S7.pdf', bbox_inches='tight')
plt.show()